# CDS Pricing

This notebook walks through the `credit` package, which prices single-name Credit Default Swaps (CDS).

A CDS is a bilateral contract where:
- The **protection buyer** pays a periodic coupon (the *spread*) and receives the loss given default (LGD = (1 − R) × notional) if the reference entity defaults before maturity.
- The **protection seller** receives the coupon and makes the LGD payment on default.

Pricing requires two market inputs:
- A **`ZeroCurve`** for risk-free discounting.
- A **`SurvivalCurve`** encoding the market's view of the reference entity's default probability, bootstrapped from quoted CDS spreads.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == 'examples' else Path.cwd()
sys.path.insert(0, str(root))

from database import set_db_path
set_db_path(str(root / 'examples' / 'demo.db'))

In [ ]:
import sqlite3
from database import get_db_path
from scripts.initialise import init_db, _seed_holidays

init_db()
with sqlite3.connect(get_db_path()) as conn:
    count = conn.execute("SELECT COUNT(*) FROM holidays").fetchone()[0]
    if count == 0:
        _seed_holidays(conn)
        print(f"Seeded demo.db with holiday data.")
    else:
        print(f"demo.db already seeded ({count} holidays). Skipping.")

In [ ]:
from datetime import date
from dateutil.relativedelta import relativedelta
import matplotlib.pyplot as plt

from market_structures import ZeroCurve
from market_conventions import DayCountConvention, CompoundingType
from credit import SurvivalCurve, SingleNameCDS

## 1. Discount Curve

We build a simple USD risk-free zero curve — a gently upward-sloping curve around 5%.

In [ ]:
REF = date(2024, 1, 2)
pillar = lambda m: REF + relativedelta(months=m)

dc = ZeroCurve(
    reference_date=REF,
    pillar_dates=[pillar(6), pillar(12), pillar(24), pillar(36), pillar(60), pillar(84), pillar(120)],
    rates=       [0.0490,   0.0500,    0.0510,    0.0520,    0.0530,    0.0525,    0.0510],
    day_count_convention=DayCountConvention.ACT_365_FIXED,
    compounding_type=CompoundingType.CONTINUOUS,
)

tenors = ["6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y"]
print(f"{'Tenor':<8} {'Zero Rate':>10} {'Discount Factor':>18}")
print("-" * 40)
for label, d in zip(tenors, dc._pillar_dates):
    print(f"{label:<8} {dc.zero_rate(d):>10.3%}  {dc.discount_factor(d):>18.6f}")

## 2. Survival Curve Bootstrap

A `SurvivalCurve` is built from quoted CDS spreads using `from_spreads()`. The bootstrap iterates over pillars in order, solving for the piecewise-constant hazard rate at each maturity via bisection so that the implied par spread matches the market quote.

We use upward-sloping spreads — typical for an investment-grade name with moderate term premium.

In [ ]:
RECOVERY = 0.40

cds_pillars = [pillar(12), pillar(24), pillar(36), pillar(60), pillar(84), pillar(120)]
market_spreads = [0.0050, 0.0065, 0.0080, 0.0100, 0.0115, 0.0130]  # 50, 65, 80, 100, 115, 130 bps

sc = SurvivalCurve.from_spreads(
    reference_date=REF,
    pillar_dates=cds_pillars,
    spreads=market_spreads,
    discount_curve=dc,
    recovery_rate=RECOVERY,
)

tenors = ["1Y", "2Y", "3Y", "5Y", "7Y", "10Y"]
print(f"{'Tenor':<8} {'Mkt Spread':>12} {'Hazard Rate':>14} {'Survival Prob':>16}")
print("-" * 55)
for label, d, s, h in zip(tenors, cds_pillars, market_spreads, sc._hazard_rates):
    q = sc.survival_probability(d)
    print(f"{label:<8} {s:>11.0f}bp  {h:>13.4%}  {q:>15.4%}")

### Survival Curve Shape

The survival probability is the probability of the reference entity *not* having defaulted by a given date. It is a monotonically decreasing function of time.

In [ ]:
monthly_dates = [REF + relativedelta(months=m) for m in range(1, 121)]
survival_probs = [sc.survival_probability(d) for d in monthly_dates]
t_axis = [(d - REF).days / 365.25 for d in monthly_dates]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_axis, [q * 100 for q in survival_probs], color='steelblue', linewidth=2)
ax.scatter(
    [(d - REF).days / 365.25 for d in cds_pillars],
    [sc.survival_probability(d) * 100 for d in cds_pillars],
    color='steelblue', s=50, zorder=5, label='CDS pillars'
)
ax.set_xlabel('Years')
ax.set_ylabel('Survival probability (%)')
ax.set_title('Bootstrapped Survival Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Bootstrap Round-Trip Verification

The bootstrapped curve should reproduce the input market spreads exactly (to numerical tolerance).

In [ ]:
print(f"{'Tenor':<8} {'Mkt Spread':>12} {'Implied Spread':>16} {'Error (bps)':>14}")
print("-" * 56)
for label, d, s_mkt in zip(tenors, cds_pillars, market_spreads):
    cds_check = SingleNameCDS(
        effective_date=REF,
        maturity_date=d,
        notional=1_000_000,
        coupon_spread=s_mkt,
        recovery_rate=RECOVERY,
    )
    implied = cds_check.par_spread(dc, sc)
    err_bps = (implied - s_mkt) * 10_000
    print(f"{label:<8} {s_mkt*10000:>11.0f}bp  {implied*10000:>15.4f}bp  {err_bps:>13.2e}")

## 4. CDS Pricing

We price a 5-year CDS with a running coupon of 100 bps (the par spread at the 5Y pillar). Since the coupon equals the par spread, the MTM should be zero.

In [ ]:
NOTIONAL = 10_000_000  # $10M
COUPON = 0.0100        # 100 bps — par spread at 5Y
MATURITY = pillar(60)  # 5 years

cds = SingleNameCDS(
    effective_date=REF,
    maturity_date=MATURITY,
    notional=NOTIONAL,
    coupon_spread=COUPON,
    recovery_rate=RECOVERY,
    is_protection_buyer=True,
)

par = cds.par_spread(dc, sc)
prot = cds.protection_leg_pv(dc, sc)
prem = cds.premium_leg_pv(dc, sc)
rv01 = cds.rpv01(dc, sc)
mtm = cds.mtm(dc, sc)

print(f"Par spread:         {par*10000:>10.4f} bps")
print(f"Protection leg PV:  {prot:>12,.0f}")
print(f"Premium leg PV:     {prem:>12,.0f}")
print(f"RPV01 (per unit N): {rv01:>12.6f} yrs")
print(f"MTM (buyer):        {mtm:>12,.2f}")

## 5. Greeks: CS01 and RR01

**CS01** (credit spread 01) measures the MTM change for a parallel 1 bp widening in CDS spreads. The survival curve is re-bootstrapped after the bump. Positive for the protection buyer — wider spreads increase the value of protection.

**RR01** (recovery rate 01) measures the MTM change for a 1% increase in the recovery rate. The hazard rates are held fixed ("sticky hazard rates"). Positive for the protection buyer — higher recovery reduces the expected protection payout, so the buyer benefits less from holding protection.

In [ ]:
cs01 = cds.cs01(dc, sc)
rr01 = cds.rr01(dc, sc)

print(f"CS01 (1 bp spread widening): {cs01:>10,.0f}")
print(f"RR01 (1% recovery increase): {rr01:>10,.0f}")
print()
print("For a $10M protection buyer position:")
print(f"  Spread widens 10 bps  → MTM change ≈ {10 * cs01:>10,.0f}")
print(f"  Recovery rises to 45% → MTM change ≈ {5 * rr01:>10,.0f}")

## 6. MTM as a Function of Spread Level

The MTM of a fixed-coupon CDS changes as market spreads move. Below we sweep the flat spread level and plot the buyer's MTM.

In [ ]:
spread_levels = [s / 10000 for s in range(30, 251, 5)]  # 30 bps to 250 bps
mtm_values = []

for s_flat in spread_levels:
    sc_flat = SurvivalCurve.from_spreads(
        reference_date=REF,
        pillar_dates=cds_pillars,
        spreads=[s_flat] * len(cds_pillars),
        discount_curve=dc,
        recovery_rate=RECOVERY,
    )
    mtm_values.append(cds.mtm(dc, sc_flat))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot([s * 10000 for s in spread_levels], [m / 1000 for m in mtm_values], color='steelblue', linewidth=2)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(COUPON * 10000, color='tomato', linewidth=1.2, linestyle=':', label=f'Coupon = {COUPON*10000:.0f} bps')
ax.set_xlabel('Flat market spread (bps)')
ax.set_ylabel('MTM ($000s)')
ax.set_title(f'MTM vs Spread — 5Y CDS, ${NOTIONAL/1e6:.0f}M, {COUPON*10000:.0f} bps coupon, protection buyer')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Pillar Management

Survival curves support the same `add_pillar` / `remove_pillar` interface as `ZeroCurve`. Modifying pillars clears the bootstrap metadata (since the curve is no longer tied to the original spread quotes).

In [ ]:
# Construct a simple 3-pillar curve manually (no bootstrap)
sc_manual = SurvivalCurve(
    reference_date=REF,
    pillar_dates=[pillar(12), pillar(36), pillar(60)],
    hazard_rates=[0.008, 0.012, 0.016],
)

q_5y_before = sc_manual.survival_probability(pillar(60))
print(f"5Y survival prob (3-pillar): {q_5y_before:.4%}")

# Insert a 2Y pillar
sc_manual.add_pillar(pillar(24), 0.010)
q_5y_after = sc_manual.survival_probability(pillar(60))
print(f"5Y survival prob (4-pillar): {q_5y_after:.4%}  (unchanged — later pillars not affected)")
print(f"Pillars: {[str(d) for d in sc_manual._pillar_dates]}")